# Advance Seeds — Train a Run

## ▶ To start: click **Runtime → Run all** (or press `Cmd/Ctrl + F9`)

Colab does **not** execute a notebook automatically when it opens — even when launched from the dashboard's *Open in Colab* button. You have to trigger the run yourself once. After that, sit back: each cell streams output, including live training metrics back to the registry dashboard.

---

This notebook trains the YOLO model for a registry run id and streams metrics + the final artifact back to Supabase.

**Before you Run all, check:**
1. **GPU runtime is set**: Runtime → Change runtime type → T4 (free) or better.
2. **You have a Supabase service-role key** — the notebook will prompt for it.
3. **Run id is in the URL** as `?run_id=...`. If not, paste it manually when the notebook prompts.

## 1. Resolve the run id from the URL

In [ ]:
from urllib.parse import urlparse, parse_qs
RUN_ID = ''
try:
    from google.colab import _message
    url = _message.blocking_request('get_url', timeout_sec=5)
    qs = parse_qs(urlparse(url).query)
    RUN_ID = qs.get('run_id', [''])[0]
except Exception as exc:
    print('Could not auto-read run_id from the URL:', exc)
print('Run id:', RUN_ID or '(none — paste below)')

In [ ]:
RUN_ID = RUN_ID or input('Run id: ').strip()
assert RUN_ID, 'A run id is required.'

## 2. Install dependencies + clone the repo

In [ ]:
%pip install -q ultralytics supabase pyyaml coremltools
import os, pathlib, shutil, subprocess
REPO_DIR = pathlib.Path('/content/advance-seeds-field-inspector-ml')
os.chdir('/content')
if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=True)
elif REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/phongsakorn-ipassion/advance-seeds-field-inspector-ml.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/phongsakorn-ipassion/advance-seeds-field-inspector-ml.git', str(REPO_DIR)], check=True)
%cd /content/advance-seeds-field-inspector-ml
%pip install -q -e .

## 3. Configure Supabase access

Paste your **service-role** key (not anon). It only lives in this notebook session.

In [ ]:
import getpass, os
os.environ['SUPABASE_URL'] = 'https://gqsxiohxokgwwugeoxmy.supabase.co'
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = getpass.getpass('SUPABASE_SERVICE_ROLE_KEY: ')
os.environ['ADVANCE_SEEDS_RUN_ID'] = RUN_ID
# The Python RegistryClient reads these aliases:
os.environ['MODEL_REGISTRY_URL'] = os.environ['SUPABASE_URL']
os.environ['MODEL_REGISTRY_SERVICE_ROLE_KEY'] = os.environ['SUPABASE_SERVICE_ROLE_KEY']

## 4. Pull run config + dataset

In [ ]:
from supabase import create_client
sb = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SERVICE_ROLE_KEY'])
run = sb.table('runs').select('*').eq('id', RUN_ID).single().execute().data
config = run['config_yaml']
print('dataset:', config.get('dataset'))
print('source_weights:', config.get('source_weights'))
print('classes:', len(config.get('classes', [])))
print('hyperparameters:', config.get('hyperparameters'))

In [ ]:
# If the dashboard uploaded the YAML to R2 (key starts with 'datasets/'),
# pull it down via the download-dataset Edge Function and rewrite DATASET
# to point at the local copy so YOLO can read it.
import os, requests, pathlib
DATASET = config.get('dataset', '')
if DATASET.startswith('datasets/'):
    print('Fetching YAML from R2:', DATASET)
    resp = sb.functions.invoke('download-dataset', invoke_options={'body': {'r2_key': DATASET}})
    body = resp if isinstance(resp, dict) else getattr(resp, 'data', None) or resp
    if hasattr(body, 'json'):
        body = body.json()
    if isinstance(body, (bytes, bytearray)):
        import json as _json; body = _json.loads(body)
    download_url = body.get('download_url') if isinstance(body, dict) else None
    if not download_url:
        raise RuntimeError(f'download-dataset returned no download_url: {body!r}')
    local_dir = pathlib.Path('configs')
    local_dir.mkdir(exist_ok=True)
    local_path = local_dir / pathlib.Path(DATASET).name
    local_path.write_bytes(requests.get(download_url, timeout=60).content)
    DATASET = str(local_path)
    config['dataset'] = DATASET
    print('YAML written to', DATASET)
else:
    print('Using local dataset reference:', DATASET)

# Sanity: peek at the YAML's path: field. If it points to a folder that
# does not exist in this Colab VM, training will fail with 'no images found'.
# In that case you still need to upload the images separately (Drive mount,
# manual scp, or a future dataset-bundle upload).
import yaml as _yaml
_doc = _yaml.safe_load(pathlib.Path(DATASET).read_text())
_root = pathlib.Path(DATASET).parent / _doc.get('path', '.')
_train = _root / _doc.get('train', 'images/train')
print('Resolved dataset root:', _root)
print('Resolved train images dir:', _train, '(exists:', _train.exists(), ')')
if not _train.exists():
    print('\n[!] Train images not found. Training will fail unless you place\n'
          '    image data at the resolved paths above. Common workaround:\n'
          '    1. Upload your dataset to Google Drive and mount it here, or\n'
          '    2. !gdown / !wget the images, or\n'
          '    3. Run training locally on a machine that already has the data.')

## 4b. Mount Google Drive + unzip dataset

If cell 10 reported that train images were not found, mount your Drive and unzip the dataset. Update `DATASET_ZIP_DRIVE_PATH` if your zip lives elsewhere. Skipped automatically once the data is on disk.

In [ ]:
from google.colab import drive
import pathlib

# Edit these if your zip lives elsewhere or your YAML's path: differs.
DATASET_ZIP_DRIVE_PATH = '/content/drive/MyDrive/advance-seeds/data/processed/dataset.zip'
# YAML's `path: ../data/processed/` resolves relative to the YAML's directory
# (configs/), so the actual on-disk path is <repo>/data/processed/.
EXTRACT_TARGET = pathlib.Path('/content/advance-seeds-field-inspector-ml/data/processed')

EXTRACT_TARGET.mkdir(parents=True, exist_ok=True)
already_populated = any(EXTRACT_TARGET.iterdir())

if already_populated:
    print(f'Dataset already extracted under {EXTRACT_TARGET}; skipping.')
else:
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    zip_path = pathlib.Path(DATASET_ZIP_DRIVE_PATH)
    if not zip_path.exists():
        raise RuntimeError(
            f'Dataset zip not found at {zip_path!s}. Upload it to Drive or update DATASET_ZIP_DRIVE_PATH above.'
        )
    print(f'Extracting {zip_path} -> {EXTRACT_TARGET}')
    !unzip -q -o '{zip_path}' -d '{EXTRACT_TARGET}'

print('\nTop-level contents under', EXTRACT_TARGET, ':')
!ls '{EXTRACT_TARGET}'
print('\nLook for images/train/, images/val/, images/test/ at this level. If you see\n'
      'a wrapper directory like advance-seeds-banana-v1/, your YAML\'s path: needs to\n'
      'point at that wrapper (e.g. path: ../data/processed/advance-seeds-banana-v1/).')

## 5. Train + stream metrics back

The Python SDK (`src/advance_seeds_ml/registry/`) writes per-epoch metrics to the `run_metrics` table during training, which appear live in the dashboard's Live tracking panel.

In [ ]:
# Real training: fetches the run row from Supabase, runs Ultralytics with
# the dashboard's hyperparameters, streams per-epoch metrics back, exports
# tflite, uploads the artifact to R2, creates the version, and finalizes
# the run. All under the existing run_id, so the dashboard's Live tracking
# updates as it goes.
!python scripts/train_for_run.py --run-id $ADVANCE_SEEDS_RUN_ID

## 6. Done

When the run finishes, switch back to the dashboard — the candidate version, final metrics, and the R2 artifact should all be visible. Promote it to staging or production from the Models screen.